# Tutorial 03 — Counter QoR autoresearch

> **EXPERIMENTAL.** Greedy AI loop that pareto-optimises QoR knobs over the same 4-bit counter from Tutorials 01-02. **Costs real money** (capped at ~$0.30 by default with opencode + Gemini Flash). Not part of the chipathon tapeout signoff path.

Where Tutorial 02 ran the agent **once** with a fixed config, Tutorial 03 runs it **multiple times** while sweeping a discrete knob space:

- `PL_TARGET_DENSITY_PCT` ∈ {60, 65, 70}
- `CLOCK_PERIOD` ∈ {50, 40, 30} ns
- `PDN_VWIDTH` ∈ {3, 5, 7} um

The `DigitalAutoresearchRunner` greedy-iterates: it picks a starting point, runs LibreLane, scores the result by FoM (die area + setup slack + power), proposes a neighbouring point, runs again, and keeps the best. Capped at 3 evaluations for this tutorial to bound cost.

**Pre-requisites** (one-time, see [`../docs/eda_agents_setup.md`](../docs/eda_agents_setup.md)):

- `gf180` container running.
- `eda-agents` pip-installed in an active venv with the `[adk]` extra.
- opencode CLI on PATH (`npm install -g opencode-ai`).
- `OPENROUTER_API_KEY` (or `GOOGLE_API_KEY` with model override) in your environment.


## Step 0 — Configuration + path guard

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

# --- Toggles --------------------------------------------------------
RUN_PIP_INSTALL = False  # only needed once per venv
RUN_PREFLIGHT   = True
RUN_STAGE       = True
RUN_DRY         = True   # construct runner, print knobs/FoM, no LLM call (~5s, free)
RUN_REAL        = False  # full 3-eval autoresearch loop (~10-15 min, ~$0.20-0.30)
RUN_INSPECT     = False  # tail program.md + results.tsv after RUN_REAL

# --- Backend & budget ----------------------------------------------
BACKEND = 'opencode'   # 'opencode' (cheap, multi-provider) | 'cc_cli' | 'adk' | 'litellm'
OPENCODE_MODEL = os.environ.get(
    'EDA_AGENTS_MODEL',
    'openrouter/google/gemini-3-flash-preview',
)
OPENCODE_CLI_PATH = 'opencode'

# Capped at 3 (eda-agents upstream demo uses 5). Each eval is one
# LibreLane run + agent context, ~3-5 minutes + ~$0.05-0.10 with Gemini Flash.
BUDGET = 3

# --- Path guard -----------------------------------------------------
try:
    NB_DIR = Path(__file__).resolve().parent
except NameError:
    NB_DIR = Path.cwd().resolve()

for _sub in ('rtl', 'librelane'):
    if not (NB_DIR / _sub).is_dir():
        raise RuntimeError(
            f'NB_DIR={NB_DIR} does not contain {_sub}/. '
            'Launch from tutorials/03_counter_autoresearch/.'
        )

REPO_ROOT = NB_DIR.parents[1]
HOST_WORKSPACE = Path.home() / 'eda' / 'designs' / 'eda_agents_counter_autoresearch'
WORK_DIR = NB_DIR / 'digital_autoresearch_results'
EDA_AGENTS_ROOT = Path(
    os.environ.get('EDA_AGENTS_ROOT', Path.home() / 'personal_exp' / 'eda-agents')
).resolve()

print(f'NB_DIR          : {NB_DIR}')
print(f'EDA_AGENTS_ROOT : {EDA_AGENTS_ROOT}  (exists: {EDA_AGENTS_ROOT.is_dir()})')
print(f'HOST_WORKSPACE  : {HOST_WORKSPACE}')
print(f'WORK_DIR        : {WORK_DIR}')
print(f'BACKEND         : {BACKEND}')
print(f'BUDGET          : {BUDGET} LibreLane evaluations')

## Step 1 — (Optional) install eda-agents

In [ ]:
import sys

if RUN_PIP_INSTALL:
    if not EDA_AGENTS_ROOT.is_dir():
        raise RuntimeError(f'EDA_AGENTS_ROOT does not exist: {EDA_AGENTS_ROOT}')
    cmd = [sys.executable, '-m', 'pip', 'install', '-e', f'{EDA_AGENTS_ROOT}[adk]']
    print('$ ' + ' '.join(cmd))
    subprocess.run(cmd, check=True)
else:
    print('(skipped -- flip RUN_PIP_INSTALL=True if `import eda_agents` fails)')

## Step 2 — Pre-flight: container + opencode + provider key

In [ ]:
if RUN_PREFLIGHT:
    out = subprocess.run(
        ['docker', 'ps', '--filter', 'name=gf180', '--format', '{{.Names}}'],
        capture_output=True, text=True, timeout=10,
    )
    container_ok = 'gf180' in out.stdout.split()

    try:
        import eda_agents  # noqa: F401
        eda_ok = True
    except ImportError:
        eda_ok = False

    opencode_path = shutil.which(OPENCODE_CLI_PATH)
    print(f'  gf180 container running : {container_ok}')
    print(f'  eda_agents importable   : {eda_ok}')
    print(f'  opencode CLI on PATH    : {opencode_path or "-- not found"}')
    print(f'  claude CLI on PATH      : {shutil.which("claude") or "-- not found"} (alt for cc_cli)')
    for key in ('OPENROUTER_API_KEY', 'GOOGLE_API_KEY', 'ZAI_API_KEY', 'ANTHROPIC_API_KEY'):
        print(f'  {key:<22}  : {"set" if os.environ.get(key) else "unset"}')

    if not container_ok:
        print('FIX: scripts/bootstrap_container.sh from repo root.')
    if not eda_ok:
        print('FIX: flip RUN_PIP_INSTALL=True in cell 0.')
    if BACKEND == 'opencode' and not opencode_path:
        print('FIX: npm install -g opencode-ai (or switch BACKEND).')
    if BACKEND == 'opencode' and not os.environ.get('OPENROUTER_API_KEY'):
        print('FIX: copy tutorials/.env.example -> tutorials/.env and fill OPENROUTER_API_KEY.')
else:
    print('(skipped -- flip RUN_PREFLIGHT=True)')

## Step 3 — Stage workspace

In [ ]:
if RUN_STAGE:
    HOST_WORKSPACE.mkdir(parents=True, exist_ok=True)
    for sub in ('rtl', 'librelane'):
        src = NB_DIR / sub
        dst = HOST_WORKSPACE / sub
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f'  staged {src.relative_to(REPO_ROOT)} -> {dst}')
    print(f'\nWorkspace ready: {HOST_WORKSPACE}')
else:
    print('(skipped -- flip RUN_STAGE=True)')

## Step 4 — Construct the runner + dry-run

Free, ~5 seconds. Prints the knob space, FoM, backend, model. No LLM call.

In [ ]:
import asyncio

if RUN_DRY:
    from eda_agents.core.designs.generic import GenericDesign
    from eda_agents.agents.digital_autoresearch import DigitalAutoresearchRunner

    design = GenericDesign(
        config_path=str(HOST_WORKSPACE / 'librelane' / 'config.yaml'),
        pdk_root=os.environ.get('PDK_ROOT') or None,
        pdk_config='gf180mcu',
    )
    kwargs = dict(design=design, backend=BACKEND, budget=BUDGET)
    if BACKEND == 'opencode':
        kwargs.update(opencode_cli_path=OPENCODE_CLI_PATH, opencode_model=OPENCODE_MODEL)
    runner = DigitalAutoresearchRunner(**kwargs)

    print(f'  design  : {design.project_name()}')
    print(f'  specs   : {design.specs_description()}')
    print(f'  FoM     : {design.fom_description()}')
    print(f'  budget  : {BUDGET} LibreLane evaluations')
    print(f'  knobs   : PL_TARGET_DENSITY_PCT x CLOCK_PERIOD x PDN_VWIDTH')
    print(f'  backend : {BACKEND}')
    if BACKEND == 'opencode':
        print(f'  model   : {OPENCODE_MODEL}')
    print(f'  WORK_DIR: {WORK_DIR}')
else:
    print('(skipped -- flip RUN_DRY=True)')

## Step 5 — Real run (~10-15 min, ~$0.20-0.30)

Each evaluation is one full LibreLane run + agent reasoning over what to try next. With BUDGET=3 and Gemini Flash, expect:

- ~3-5 minutes per LibreLane run on this counter
- ~$0.05-0.10 per agent turn (the agent emits one decision + a short rationale)
- Total: ~10-15 minutes wall time, ~$0.20-0.30 LLM spend

Live progress lands under `WORK_DIR`: `program.md` accumulates the agent's decision log, `results.tsv` accumulates the FoM scores per knob combo.

In [ ]:
import json

if RUN_REAL:
    from eda_agents.core.designs.generic import GenericDesign
    from eda_agents.agents.digital_autoresearch import DigitalAutoresearchRunner

    design = GenericDesign(
        config_path=str(HOST_WORKSPACE / 'librelane' / 'config.yaml'),
        pdk_root=os.environ.get('PDK_ROOT') or None,
        pdk_config='gf180mcu',
    )
    kwargs = dict(design=design, backend=BACKEND, budget=BUDGET)
    if BACKEND == 'opencode':
        kwargs.update(opencode_cli_path=OPENCODE_CLI_PATH, opencode_model=OPENCODE_MODEL)
    runner = DigitalAutoresearchRunner(**kwargs)

    async def _real():
        return await runner.run(WORK_DIR)

    result = asyncio.run(_real())
    safe = {k: v for k, v in result.__dict__.items() if not k.startswith('_')}
    print(json.dumps(safe, indent=2, default=str))
else:
    print('(skipped -- flip RUN_REAL=True after dry-run + env are clean)')
    print(f'Expected: ~10-15 min, ~$0.20-0.30, artifacts under {WORK_DIR}')

## Step 6 — Inspect program.md + results.tsv

In [ ]:
if RUN_INSPECT:
    for fname in ('program.md', 'results.tsv'):
        p = WORK_DIR / fname
        if p.exists():
            print(f'--- {p} ---')
            print(p.read_text()[:3000])
            print()
        else:
            print(f'{p} not yet written; run step 5 first.')
else:
    print('(skipped -- flip RUN_INSPECT=True after step 5 finishes)')

## What you learned

- `DigitalAutoresearchRunner` is a greedy optimiser: it iterates the agent over a discrete knob space and picks the best by FoM.
- The default knob space is small (3x3x3 = 27 points) so a budget of 3 covers ~10% of the space — useful for quick dirs/QoR exploration, not for global optimum search.
- `BACKEND="opencode"` is the cheapest path because opencode supports any provider via OpenRouter / direct keys. Gemini Flash (~$0.075 / 1M tokens) is plenty smart for knob picking on a counter.
- The agent log (`program.md`) makes the search transparent: you see *why* it picked each next point, not just the FoM table.
- Cost cap is implicit through `BUDGET=N` (each eval is roughly bounded). For tighter budget control, drop BUDGET, drop the knob count, or switch to a cheaper model.

## Cleanup

```bash
rm -rf ~/eda/designs/eda_agents_counter_autoresearch/
rm -rf tutorials/03_counter_autoresearch/digital_autoresearch_results/
```

## Where to go from here

- Bump `BUDGET` to 5-10 if you have headroom and want a more thorough sweep.
- Add knobs by editing the FoM in `eda_agents/agents/digital_autoresearch.py` (or subclass `DigitalAutoresearchRunner`).
- Switch the design to `examples/04_counter_alu_multimacro/` (multi-macro chip-top) — the runner works on any `DigitalDesign`, including hardened-macro chips.
- Run the same loop with `BACKEND="cc_cli"` if you have a Claude subscription and want zero-marginal-cost iteration.